[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S06/2_Practica/para_casa/S06_03_preprocesado_pipeline.ipynb)

# S06 · 03 — Preprocesado y *pipelines* reproducibles

Sprint 6 · M2 (UD 2.5) + M3 (UD 3.1)

**Contexto TurisData.** Ya sabemos limpiar `reservas_sucio.csv`. Ahora hay que **preparar las columnas para un modelo**: rellenar nulos, poner las variables en la misma escala y convertir las categorías en números. Si lo hacemos a mano, tarde o temprano cometeremos un error de fuga. La solución de scikit-learn es el **`Pipeline`**: una receta única, ajustada solo con los datos de entrenamiento y aplicable a datos nuevos.

**Este cuaderno tiene 5 ejercicios y un mini-reto.** Los ejercicios 2, 4 y 5 son los mismos que los del cuaderno «en clase» (allí, del 1 al 3).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.base import clone

SEMILLA = 42
sns.set_theme(style="whitegrid")
print("Listo ✔")

### 👀 A2 · EJEMPLO — Limpiar y cargar las reservas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas filas y columnas tiene `reservas`? ¿Cuántas columnas más que el fichero sucio (18)?
2. ¿Qué columnas tienen nulos y cuántos?
3. ¿Por qué se puede limpiar antes de separar entrenamiento y prueba sin que haya fuga?

In [ ]:
EQUIV_PAIS = {"es": "ES", "españa": "ES", "uk": "UK", "reino unido": "UK", "de": "DE",
              "nl": "NL", "it": "IT", "fr": "FR", "nordicos": "Nordicos", "otros": "otros"}

def limpiar_reservas(tabla):
    t = tabla.copy()
    resumen = {"filas_antes": len(t)}
    t = t.drop_duplicates().reset_index(drop=True)
    resumen["duplicados_quitados"] = resumen["filas_antes"] - len(t)
    t["precio_noche"] = pd.to_numeric(t["precio_noche"].astype(str).str.replace(",", "."), errors="coerce")
    iso = pd.to_datetime(t["fecha_reserva"], format="%Y-%m-%d", errors="coerce")
    esp = pd.to_datetime(t["fecha_reserva"], format="%d/%m/%Y", errors="coerce")
    t["fecha_reserva"] = iso.fillna(esp)
    t["fecha_llegada"] = pd.to_datetime(t["fecha_llegada"], format="%Y-%m-%d")
    t["canal"] = t["canal"].str.strip().str.lower().str.replace(" ", "_")
    t["pais_origen"] = t["pais_origen"].str.strip().str.lower().map(EQUIV_PAIS)
    nulos_antes = t[["precio_noche", "noches"]].isna().sum().sum()
    t["precio_noche"] = t["precio_noche"].where(t["precio_noche"].between(20, 1000))
    t["noches"] = t["noches"].where(t["noches"].between(1, 60))
    resumen["imposibles_a_nulo"] = int(t[["precio_noche", "noches"]].isna().sum().sum() - nulos_antes)
    antes = t["antelacion_dias"]
    t["antelacion_dias"] = (t["fecha_llegada"] - t["fecha_reserva"]).dt.days
    resumen["antelaciones_recalculadas"] = int((t["antelacion_dias"] != antes).sum())
    t["mes_llegada"] = t["fecha_llegada"].dt.month
    resumen["filas_despues"] = len(t)
    return t, resumen

reservas, resumen = limpiar_reservas(pd.read_csv(dato("reservas_sucio.csv")))
print(resumen)
print(reservas.shape)
reservas.isna().sum()[lambda s: s > 0]

### 👀 A3 · EJEMPLO — Definir X e y

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Cuántas columnas tiene `X` y cuántas son numéricas y cuántas categóricas?
2. ¿Qué proporción de reservas se cancela?
3. ¿Por qué `cancelada` no está dentro de `X`?

In [ ]:
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "mes_llegada"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X = reservas[columnas_num + columnas_cat]
y = reservas["cancelada"]
print("X:", X.shape, "· y:", y.shape)
print("Proporción de canceladas:", y.mean().round(3))

### ✏️ A4 · EJERCICIO 1 de 5 — Separar entrenamiento y prueba

**Tu turno:** Separa `X` e `y` en `X_train, X_test, y_train, y_test` con un 20 % para prueba, `random_state=SEMILLA` y estratificando por `y`.

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Cuánto valen `len(X_train)` y `len(X_test)`?
2. ¿Qué pasaría si quitaras `random_state=SEMILLA`?
3. ¿Para qué sirve `stratify=y`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
# pista: train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
X_train, X_test, y_train, y_test = ...

assert len(X_train) == 1200 and len(X_test) == 300, "Con 1500 filas y test_size=0.2 deben ser 1200 / 300"
assert abs(y_train.mean() - y_test.mean()) < 0.01, "Estratifica con stratify=y para conservar la proporción"
print("✔ Ejercicio 1 correcto")

### 👀 A5 · EJEMPLO — Imputar y escalar con los datos de entrenamiento

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Cuánto valen las medias de las 9 columnas en train y por qué?
2. ¿Y en test? ¿Por qué no salen exactamente 0?
3. ¿Qué pasaría si hicieras `fit_transform` también con test?

In [ ]:
imputador = SimpleImputer(strategy="median")
escalador = StandardScaler()
num_train = escalador.fit_transform(imputador.fit_transform(X_train[columnas_num]))
num_test = escalador.transform(imputador.transform(X_test[columnas_num]))
print("Media de cada columna en train (≈0):", num_train.mean(axis=0).round(2))
print("Media en test (no tiene por qué ser 0):", num_test.mean(axis=0).round(2))

### ✏️ A6 · EJERCICIO 2 de 5 — Escalar el precio con MinMaxScaler

**Tu turno:** Crea un `MinMaxScaler` llamado `esc_minmax`, ajústalo **solo con `X_train[["precio_noche"]].dropna()`** y guarda en `precio_test_esc` el precio de test transformado (rellena antes los nulos de test con la mediana de train: `mediana_precio`).

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Entre qué valores se ajustó el escalador y cuánto vale la mediana de train?
2. `precio_test_esc` va de 0,025 a 1,006: ¿por qué se sale de 0–1?
3. La media de `precio_test_esc` es 0,278: ¿qué significa?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
# pista: MinMaxScaler(); .fit con train (sin nulos); la mediana de precio también sale de train
esc_minmax = ...
mediana_precio = ...
precio_test_esc = ...

assert esc_minmax.data_min_[0] == X_train["precio_noche"].min(), "El escalador debe ajustarse con train, no con test ni con todo"
assert precio_test_esc.shape == (300, 1)
assert mediana_precio == X_train["precio_noche"].median(), "La mediana debe salir de train"
print("✔ Ejercicio 2 correcto")

### 👀 A7 · EJEMPLO — Codificar el canal en one-hot

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Cuántas columnas crea el codificador para `canal` y cómo se llaman?
2. ¿Por qué one-hot para el canal y no 0, 1, 2, 3?
3. ¿Qué hace `handle_unknown="ignore"` si en test aparece un canal nuevo?

In [ ]:
onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
onehot.fit(X_train[["canal"]])
print(onehot.get_feature_names_out())

### ✏️ A8 · EJERCICIO 3 de 5 — Codificar el depósito con orden

**Tu turno:** Codifica `deposito` con un `OrdinalEncoder` que respete el orden `["sin_deposito", "parcial", "total"]`. Guarda el encoder en `ord_dep` y el resultado de train en `dep_train_cod`.

**Debe salir:** ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A8).

1. ¿Qué número recibe cada depósito?
2. ¿Cuántas reservas de train hay con cada valor?
3. ¿Qué pasaría si no pusieras `categories=`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A8, **después de intentarlo**.

In [ ]:
# pista: OrdinalEncoder(categories=[[...]]) y fit_transform sobre X_train[["deposito"]]
ord_dep = ...
dep_train_cod = ...

assert set(np.unique(dep_train_cod)) == {0.0, 1.0, 2.0}, "Deben salir los valores 0, 1 y 2"
assert ord_dep.transform(pd.DataFrame({"deposito": ["total"]}))[0, 0] == 2.0, "'total' debe valer 2"
print("✔ Ejercicio 3 correcto")

### ✏️ A9 · EJERCICIO 4 de 5 — Construir el ColumnTransformer

**Tu turno:** Construye `preprocesado` (un `ColumnTransformer`) con dos ramas: `"num"` (imputador de mediana + `StandardScaler`) sobre `columnas_num`, y `"cat"` (imputador `most_frequent` + `OneHotEncoder(handle_unknown="ignore")`) sobre `columnas_cat`.

**Debe salir:** `Columnas tras el preprocesado: 34` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Cuántas columnas tiene `matriz` y por qué son más que las 15 de `X`?
2. ¿Cuántas filas tiene `matriz` y por qué?
3. ¿Por qué hay dos ramas y no un único `StandardScaler` para todo?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
# pista: ColumnTransformer([("num", Pipeline([...]), columnas_num), ("cat", Pipeline([...]), columnas_cat)])
preprocesado = ...
matriz = preprocesado.fit_transform(X_train)

assert matriz.shape[0] == 1200, "Debe transformar las 1200 filas de train"
assert not np.isnan(matriz if isinstance(matriz, np.ndarray) else matriz.toarray()).any(), "Tras el preprocesado no deben quedar nulos"
assert matriz.shape[1] > len(columnas_num) + len(columnas_cat), "Las categóricas deben expandirse en columnas one-hot"
print("Columnas tras el preprocesado:", matriz.shape[1])
print("✔ Ejercicio 4 correcto")

### ✏️ A10 · EJERCICIO 5 de 5 — Pipeline con el modelo

**Tu turno:** Crea `modelo`, un `Pipeline` con los pasos `("prep", preprocesado)` y `("clf", LogisticRegression(max_iter=1000, random_state=SEMILLA))`. Ajústalo con train y guarda en `acc_test` su exactitud (`score`) en test.

**Debe salir:** `Exactitud en test: 0.680  (referencia «siempre la clase mayoritaria»: 0.530)` y ✔ Ejercicio 5 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A10).

1. ¿Qué exactitud sale y con qué referencia se compara?
2. ¿Qué ocurre con el preprocesado al llamar a `modelo.fit(X_train, y_train)`?
3. ¿Por qué `modelo.score(X_test, y_test)` acepta las columnas «sin preparar»?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A10, **después de intentarlo**.

In [ ]:
# pista: Pipeline([("prep", preprocesado), ("clf", LogisticRegression(...))]); .fit(X_train, y_train); .score(X_test, y_test)
modelo = ...
acc_test = ...

assert hasattr(modelo, "predict"), "modelo debe ser un Pipeline"
assert acc_test > 0.60, "La exactitud debería superar 0,60; revisa el pipeline"
print(f"Exactitud en test: {acc_test:.3f}  (referencia «siempre la clase mayoritaria»: {max(y_test.mean(), 1 - y_test.mean()):.3f})")
print("✔ Ejercicio 5 correcto")

### 👀 A11 · EJEMPLO — Escalar con todos los datos o solo con train

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A11).

1. ¿Cuánto difieren las dos medias?
2. Si la diferencia es tan pequeña, ¿por qué importa?

In [ ]:
# MAL: el escalador "ve" también las filas de test
esc_mal = StandardScaler().fit(reservas[["precio_noche"]].fillna(reservas["precio_noche"].median()))
# BIEN: solo train
esc_bien = StandardScaler().fit(X_train[["precio_noche"]].fillna(X_train["precio_noche"].median()))
print("Media aprendida (mal, con todo):", esc_mal.mean_[0].round(3))
print("Media aprendida (bien, solo train):", esc_bien.mean_[0].round(3))

### 👀 A12 · EJEMPLO — Varianza explicada por PCA

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Qué componente explica más variación y cuánta, aproximadamente?
2. ¿Qué parte de la variación recogen entre las cinco?
3. ¿Por qué las barras son tan parecidas entre sí?

In [ ]:
pca = PCA(n_components=5, random_state=SEMILLA).fit(num_train)
plt.figure(figsize=(6, 3.5))
plt.bar(range(1, 6), pca.explained_variance_ratio_, color="#2a6f97")
plt.title("Varianza explicada por componente (PCA)")
plt.xlabel("Componente")
plt.ylabel("Proporción de la varianza")
plt.show()

### ✏️ A13 · MINI-RETO — Pipeline reproducible

**Tu turno:** Un pipeline reproducible da **las mismas predicciones** si lo vuelves a ajustar con los mismos datos. Crea `modelo2` (otra instancia del mismo pipeline usando `clone`), ajústalo y guarda en `son_iguales` un booleano que diga si las predicciones de test coinciden con las de `modelo`.

**Debe salir:** ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A13).

1. ¿Qué vale `son_iguales` y por qué?
2. ¿Qué hace `clone` y por qué no basta con `modelo2 = modelo`?

In [ ]:
# pista: modelo2 = clone(modelo).fit(...); compara las predicciones con np.array_equal
modelo2 = ...
son_iguales = ...

assert son_iguales, "Las predicciones deberían coincidir: comprueba que fijas la semilla y ajustas con los mismos datos"
assert modelo2 is not modelo, "modelo2 debe ser una instancia nueva (usa clone)"
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga las herramientas: `pandas` (tablas), `numpy` (números), `matplotlib` y `seaborn` (gráficos) y, de `scikit-learn`, todo lo del preprocesado (`train_test_split`, `SimpleImputer`, escaladores, codificadores, `ColumnTransformer`, `Pipeline`, `LogisticRegression`, `PCA`, `clone`).
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - `SEMILLA = 42` fija el azar para que todos obtengáis lo mismo.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Punto de partida: datos ya limpiados

#### A2 · Ejemplo — Limpiar y cargar las reservas

`limpiar_reservas` es la función de la práctica anterior (S06_02), aquí te la damos hecha y con una línea más: `mes_llegada`. Solo aplica **reglas fijas** (quitar duplicados, arreglar formatos, pasar los imposibles a nulo…): no calcula medias ni medianas, es decir, **no aprende nada de los datos**. Por eso puede ejecutarse antes de separar entrenamiento y prueba sin riesgo de fuga.

- **Qué hace:**
    - `EQUIV_PAIS` es el diccionario que unifica las formas de escribir cada país.
    - `def limpiar_reservas(tabla):` junta todos los pasos de limpieza y devuelve la tabla limpia y un `resumen` de lo que ha cambiado.
    - `pd.read_csv(dato("reservas_sucio.csv"))` lee el fichero sucio y se lo pasa a la función; el resultado queda en `reservas` y `resumen`.
    - `reservas.isna().sum()[lambda s: s > 0]` cuenta los nulos de cada columna y enseña solo las que tienen alguno.
- **Qué debe salir:**
    - El diccionario `{'filas_antes': 1545, 'duplicados_quitados': 45, 'imposibles_a_nulo': 14, 'antelaciones_recalculadas': 5, 'filas_despues': 1500}`.
    - `(1500, 19)`.
    - Una lista con cinco columnas y sus nulos: `noches` 6, `adultos` 46, `regimen` 59, `pais_origen` 112 y `precio_noche` 87.
- **Si falla:**
    - `FileNotFoundError` → no has ejecutado la celda de preparación (`dato(...)` la define).
    - `NameError: name 'pd' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas filas y columnas tiene `reservas`? ¿Cuántas columnas más que el fichero sucio (18)?* → **1500 filas y 19 columnas**: una más, `mes_llegada`.
2. *¿Qué columnas tienen nulos y cuántos?* → `precio_noche` 87, `pais_origen` 112, `regimen` 59, `adultos` 46 y `noches` 6. Estos huecos hay que rellenarlos, pero **dentro del pipeline**, con datos de entrenamiento.
3. *¿Por qué se puede limpiar antes de separar entrenamiento y prueba sin que haya fuga?* → Porque la limpieza usa **reglas fijas** (quitar copias, cambiar formatos, pasar a nulo lo imposible): no calcula nada a partir de los datos, así que no puede «colar» información de test.

</details>

## 2. Separar entrenamiento y prueba

#### A3 · Ejemplo — Definir X e y

`X` son las **variables de entrada** (lo que sabemos de la reserva) e `y` la **respuesta** que queremos predecir: si la reserva se cancela (1) o no (0).

- **Qué hace:**
    - `columnas_num` y `columnas_cat` listan las columnas numéricas y las categóricas: cada tipo recibirá su propio tratamiento.
    - `X = reservas[columnas_num + columnas_cat]` se queda solo con esas 15 columnas; `y = reservas["cancelada"]` es la respuesta.
    - Las dos líneas de `print` enseñan los tamaños y la proporción de canceladas (la media de una columna de 0 y 1 es la proporción de unos).
- **Qué debe salir:**
    - `X: (1500, 15) · y: (1500,)`.
    - `Proporción de canceladas: 0.471`.
- **Si falla:**
    - `NameError: name 'reservas' is not defined` → falta ejecutar la celda anterior (sección 1).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas columnas tiene `X` y cuántas son numéricas y cuántas categóricas?* → **15**: 9 numéricas y 6 categóricas.
2. *¿Qué proporción de reservas se cancela?* → **0,471**: casi la mitad (706 de 1500). Un modelo tonto que dijera siempre «no se cancela» acertaría ≈ 0,53.
3. *¿Por qué `cancelada` no está dentro de `X`?* → Porque es justo lo que queremos predecir: si estuviera en `X`, el modelo «copiaría» la respuesta y no aprendería nada útil.

</details>

#### A4 · Ejercicio 1 — Separar entrenamiento y prueba

**Idea.** Guardamos una parte de los datos (p. ej. el 20 %) que el modelo **nunca verá** durante el ajuste: es el examen final. Con `stratify=y` se mantiene la misma proporción de canceladas en ambos conjuntos. `random_state` fija la semilla para poder repetirlo.

- **Qué hace:**
    - `train_test_split` devuelve **cuatro** cosas, en este orden: entrenamiento de `X`, prueba de `X`, entrenamiento de `y` y prueba de `y`.
    - `test_size=0.2` reserva el 20 % para prueba; `random_state=SEMILLA` fija el reparto; `stratify=y` mantiene la proporción de canceladas.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`. `X_train` tiene 1200 filas y `X_test` 300.
- **Si falla:**
    - `Con 1500 filas y test_size=0.2 deben ser 1200 / 300` → falta `test_size=0.2` o no has reasignado las cuatro variables.
    - `Estratifica con stratify=y para conservar la proporción` → falta `stratify=y`.
    - `NameError: name 'X' is not defined` → falta ejecutar las celdas anteriores.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `len(X_train)` y `len(X_test)`?* → **1200 y 300.**
2. *¿Qué pasaría si quitaras `random_state=SEMILLA`?* → Cada ejecución haría un reparto distinto y los resultados **no serían repetibles**: tú y yo obtendríamos números diferentes.
3. *¿Para qué sirve `stratify=y`?* → Para que entrenamiento y prueba tengan la misma proporción de canceladas (0,471 en train y 0,47 en test). Sin ello, por azar, uno podría tener muchas más que el otro.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Escalado

#### A5 · Ejemplo — Imputar y escalar con los datos de entrenamiento

**Idea.** `precio_noche` va de 50 a 250 y `ninos` de 0 a 3. Los modelos basados en distancias o en gradiente (regresión logística, k-NN, SVM, redes) se «fijan» en las columnas grandes. Escalar pone todas en una escala comparable:
- `StandardScaler`: resta la media y divide por la desviación (media 0, desviación 1).
- `MinMaxScaler`: lleva todo al rango 0–1 (sensible a valores extremos).

**Se ajusta (`fit`) con train y se aplica (`transform`) a train y a test.**

- **Qué hace:**
    - `SimpleImputer(strategy="median")` rellena los nulos de cada columna con su **mediana**; `StandardScaler()` resta la media y divide por la desviación.
    - `fit_transform` sobre `X_train[columnas_num]` **aprende** (mediana, media, desviación) con train y lo aplica. Hay que hacerlo primero el imputador y después el escalador.
    - En test se usa solo `transform`: se aplica **lo aprendido en train**, sin volver a calcular nada.
    - Los dos `print` enseñan la media de cada columna ya escalada, en train y en test (`.round(2)` la redondea a 2 decimales).
- **Qué debe salir:**
    - `Media de cada columna en train (≈0): [-0. -0. -0.  0.  0. -0.  0.  0. -0.]`: nueve valores, todos 0 (el `-0.` es un cero negativo, sin importancia).
    - `Media en test (no tiene por qué ser 0): [ 0.16 -0.01  0.    0.06  0.01  0.01 -0.07 -0.1  -0.01]`.
- **Si falla:**
    - `NameError: name 'X_train' is not defined` → falta ejecutar la celda de separar entrenamiento y prueba (sección 2). En la versión completa, hacer antes el ejercicio 1.
    - `ValueError: Input contains NaN` → has quitado el imputador: el escalador no admite nulos de entrada.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen las medias de las 9 columnas en train y por qué?* → **≈ 0** en todas: el escalador resta a cada columna la media que aprendió en train.
2. *¿Y en test? ¿Por qué no salen exactamente 0?* → Salen entre −0,1 y 0,16 (la mayor es `antelacion_dias`, 0,16). A test se le resta la media **de train**, no la suya, y es lo correcto: son datos «nuevos» y su media real es un poco distinta.
3. *¿Qué pasaría si hicieras `fit_transform` también con test?* → Test tendría media 0 *por construcción*, pero se habría escalado con **sus propios** estadísticos: eso es fuga, y en producción (una reserva suelta) no se podría hacer.

</details>

#### A6 · Ejercicio 2 — Escalar el precio con MinMaxScaler

- **Qué hace:**
    - `MinMaxScaler()` crea el escalador que lleva los valores al rango 0–1: el mínimo de train pasa a 0 y el máximo a 1.
    - `.fit(X_train[["precio_noche"]].dropna())` aprende el mínimo y el máximo **solo con train** y sin nulos. Con dos corchetes `[[...]]` se pasa una tabla de una columna, que es lo que pide scikit-learn.
    - `mediana_precio` se calcula con train (`X_train["precio_noche"].median()`).
    - Los nulos de test se rellenan con esa mediana (`.fillna(mediana_precio)`) y entonces se transforma (`.transform`), sin volver a ajustar.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`. `esc_minmax` aprendió mínimo 52,7 y máximo 231,78, `mediana_precio` vale 93,65 y `precio_test_esc` tiene forma `(300, 1)`.
- **Si falla:**
    - `El escalador debe ajustarse con train, no con test ni con todo` → has hecho `fit` con otra tabla (o con `fit_transform` sobre test).
    - `assert precio_test_esc.shape == (300, 1)` → falta transformar con `X_test[["precio_noche"]]` (con doble corchete) o has perdido filas.
    - `La mediana debe salir de train` → la has calculado con `reservas` o con `X_test`.
    - `ValueError: Input contains NaN` → faltó `.fillna(mediana_precio)` antes de `transform`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Entre qué valores se ajustó el escalador y cuánto vale la mediana de train?* → Mínimo **52,7**, máximo **231,78** y mediana **93,65** €.
2. *`precio_test_esc` va de 0,025 a 1,006: ¿por qué se sale de 0–1?* → Porque en test hay un precio **mayor que el máximo de train** (231,78): se transforma con lo aprendido en train, así que puede pasarse de 1 (o bajar de 0). No es un error.
3. *La media de `precio_test_esc` es 0,278: ¿qué significa?* → Que la mayoría de los precios están en el **tercio bajo** del rango (0 = precio más barato de train, 1 = más caro).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
esc_minmax = MinMaxScaler()
esc_minmax.fit(X_train[["precio_noche"]].dropna())
mediana_precio = X_train["precio_noche"].median()
precio_test_esc = esc_minmax.transform(X_test[["precio_noche"]].fillna(mediana_precio))
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Codificación de categóricas

#### A7 · Ejemplo — Codificar el canal en one-hot

**Idea.** Los modelos solo entienden números.
- **One-hot**: una columna 0/1 por cada categoría. Para categorías **sin orden** (canal, país). Con `handle_unknown="ignore"` no falla si en test aparece una categoría nueva.
- **Ordinal**: 0, 1, 2… respetando un **orden real** (depósito: sin < parcial < total).

No uses ordinal en categorías sin orden: el modelo creería que «UK» está «entre» «ES» y «DE».

- **Qué hace:**
    - `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` crea el codificador: una columna por categoría, con 1 donde la reserva es de esa categoría y 0 en las demás.
    - `.fit(X_train[["canal"]])` aprende **qué categorías existen** mirando solo train.
    - `get_feature_names_out()` enseña los nombres de las columnas nuevas.
- **Qué debe salir:**
    - `['canal_agencia' 'canal_directo' 'canal_empresa' 'canal_web_intermediaria']`.
- **Si falla:**
    - `NameError: name 'X_train' is not defined` → falta ejecutar la celda de separar entrenamiento y prueba (sección 2).
    - `TypeError: ... sparse_output` → versión antigua de scikit-learn (usaba `sparse=False`). En Colab no debería pasar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas columnas crea el codificador para `canal` y cómo se llaman?* → **4**: `canal_agencia`, `canal_directo`, `canal_empresa` y `canal_web_intermediaria` (una por canal).
2. *¿Por qué one-hot para el canal y no 0, 1, 2, 3?* → Porque los canales **no tienen orden**: con números, el modelo creería que `web_intermediaria` «vale más» que `agencia`.
3. *¿Qué hace `handle_unknown="ignore"` si en test aparece un canal nuevo?* → Lo codifica con **ceros en todas las columnas** y sigue; sin esa opción daría un error.

</details>

#### A8 · Ejercicio 3 — Codificar el depósito con orden

- **Qué hace:**
    - `OrdinalEncoder(categories=[["sin_deposito", "parcial", "total"]])` convierte cada categoría en su posición en esa lista: 0, 1 y 2. Los **dobles corchetes** son porque se pasa una lista de categorías por cada columna.
    - `fit_transform(X_train[["deposito"]])` lo ajusta con train y devuelve los valores ya codificados.
- **Qué debe salir:**
    - `✔ Ejercicio 3 correcto`. `dep_train_cod` solo contiene 0, 1 y 2 (667 reservas con 0, 361 con 1 y 172 con 2) y `'total'` vale 2.
- **Si falla:**
    - `Deben salir los valores 0, 1 y 2` → falta `fit_transform` sobre `X_train[["deposito"]]` o el orden de `categories` no incluye las tres.
    - `'total' debe valer 2` → has puesto las categorías en otro orden: `total` debe ir **la última**.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué número recibe cada depósito?* → `sin_deposito` → **0**, `parcial` → **1** y `total` → **2**.
2. *¿Cuántas reservas de train hay con cada valor?* → **667** con 0, **361** con 1 y **172** con 2 (suman las 1200). Casi la mitad no pagó depósito.
3. *¿Qué pasaría si no pusieras `categories=`?* → El codificador ordenaría las categorías **alfabéticamente**: `parcial` 0, `sin_deposito` 1, `total` 2. Es un orden falso (¡no pagar nada valdría más que pagar una parte!).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
ord_dep = OrdinalEncoder(categories=[["sin_deposito", "parcial", "total"]])
dep_train_cod = ord_dep.fit_transform(X_train[["deposito"]])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. `ColumnTransformer` y `Pipeline`: la receta completa

#### A9 · Ejercicio 4 — Construir el ColumnTransformer

**Idea.** Cada tipo de columna necesita su propio tratamiento:
- numéricas → imputar mediana → escalar,
- categóricas → imputar la más frecuente → one-hot.

`ColumnTransformer` aplica cada receta a sus columnas y `Pipeline` encadena el preprocesado con el modelo. Al llamar a `.fit(X_train, y_train)` **todo** se ajusta solo con train.

- **Qué hace:**
    - `ColumnTransformer([...])` recibe una lista de ramas: cada una es `(nombre, receta, columnas)`.
    - La rama `"num"` es un `Pipeline` de dos pasos (imputar con la mediana → `StandardScaler`) sobre `columnas_num`; la rama `"cat"`, otro de dos pasos (imputar la más frecuente → `OneHotEncoder(handle_unknown="ignore")`) sobre `columnas_cat`.
    - `preprocesado.fit_transform(X_train)` aprende y aplica todo con train, y el resultado queda en `matriz`.
- **Qué debe salir:**
    - `Columnas tras el preprocesado: 34`.
    - `✔ Ejercicio 4 correcto`.
- **Si falla:**
    - `Debe transformar las 1200 filas de train` → `fit_transform` no se aplica a `X_train`.
    - `Tras el preprocesado no deben quedar nulos` → falta algún imputador en una de las dos ramas.
    - `Las categóricas deben expandirse en columnas one-hot` → falta el `OneHotEncoder` en la rama `"cat"`.
    - `NameError: name 'columnas_num' is not defined` → falta ejecutar la celda «Definir X e y».

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas columnas tiene `matriz` y por qué son más que las 15 de `X`?* → **34** = 9 numéricas + 25 de one-hot (los valores distintos de las 6 categóricas: 4 + 4 + 4 + 8 + 2 + 3 = 25).
2. *¿Cuántas filas tiene `matriz` y por qué?* → **1200**: es la parte de entrenamiento, no las 1500.
3. *¿Por qué hay dos ramas y no un único `StandardScaler` para todo?* → Porque cada tipo necesita **su receta**: los números se imputan con la mediana y se escalan; las categorías se imputan con la más frecuente y se convierten en columnas 0/1. Escalar texto no tiene sentido.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
preprocesado = ColumnTransformer([
    ("num", Pipeline([("imputar", SimpleImputer(strategy="median")),
                      ("escalar", StandardScaler())]), columnas_num),
    ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                      ("codificar", OneHotEncoder(handle_unknown="ignore"))]), columnas_cat),
])
matriz = preprocesado.fit_transform(X_train)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A10 · Ejercicio 5 — Pipeline con el modelo

- **Qué hace:**
    - `Pipeline([("prep", preprocesado), ("clf", LogisticRegression(max_iter=1000, random_state=SEMILLA))])` encadena dos pasos: primero el preprocesado y luego el clasificador.
    - `modelo.fit(X_train, y_train)` ajusta **todo** el pipeline con train: imputadores, escalador, codificador y regresión logística.
    - `modelo.score(X_test, y_test)` prepara los datos de test con lo aprendido en train, predice y devuelve la **exactitud** (proporción de aciertos).
- **Qué debe salir:**
    - `Exactitud en test: 0.680  (referencia «siempre la clase mayoritaria»: 0.530)`.
    - `✔ Ejercicio 5 correcto`.
- **Si falla:**
    - `modelo debe ser un Pipeline` → `modelo` no es el resultado de `Pipeline([...])`: ojo, `.fit(...)` devuelve el propio pipeline, pero si lo asignas con otra cosa se pierde.
    - `La exactitud debería superar 0,60` → algo falla en el pipeline: ¿has ajustado con train y puntuado con test? ¿está `preprocesado` dentro?
    - `NameError: name 'preprocesado' is not defined` → falta el ejercicio anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué exactitud sale y con qué referencia se compara?* → **0,680** frente a **0,530** de «decir siempre la clase mayoritaria»: el modelo aporta 15 puntos.
2. *¿Qué ocurre con el preprocesado al llamar a `modelo.fit(X_train, y_train)`?* → Que se **ajusta solo con train** (medianas, medias, categorías) y después se entrena el clasificador. Nada de test interviene.
3. *¿Por qué `modelo.score(X_test, y_test)` acepta las columnas «sin preparar»?* → Porque el pipeline aplica por dentro el preprocesado (con `transform`) antes de predecir: el cliente puede darle datos crudos.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
modelo = Pipeline([("prep", preprocesado),
                   ("clf", LogisticRegression(max_iter=1000, random_state=SEMILLA))])
modelo.fit(X_train, y_train)
acc_test = modelo.score(X_test, y_test)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 6. Fuga de datos en el preprocesado

#### A11 · Ejemplo — Escalar con todos los datos o solo con train

Hay **fuga** cuando el modelo «ve» información del conjunto de prueba durante el ajuste. Un caso típico: escalar o imputar **todo** el conjunto antes de separar. Los estadísticos (media, mediana) ya contienen información de test. El efecto suele ser pequeño en un escalado, pero el hábito es peligroso: con otras transformaciones puede inflar mucho las métricas. Con un `Pipeline` el problema desaparece.

- **Qué hace:**
    - `esc_mal` aprende la media del precio con **todas** las reservas (`reservas`), también las de test.
    - `esc_bien` la aprende solo con `X_train`. En los dos casos los nulos se rellenan antes con la mediana correspondiente.
    - Los `print` enseñan la media que aprendió cada uno (`.mean_[0]`, redondeada a 3 decimales).
- **Qué debe salir:**
    - `Media aprendida (mal, con todo): 102.179`.
    - `Media aprendida (bien, solo train): 102.144`.
- **Si falla:**
    - `NameError: name 'X_train' is not defined` → falta ejecutar la celda de separar entrenamiento y prueba.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto difieren las dos medias?* → Muy poco: 102,179 frente a 102,144, apenas **0,035 €**.
2. *Si la diferencia es tan pequeña, ¿por qué importa?* → Porque es un **mal hábito**: con otras transformaciones (selección de variables, imputaciones más complejas…) la fuga puede inflar mucho las métricas, y en producción no existirá un «todo» que mirar. La regla es siempre: `fit` solo con train.

</details>

## 7. Un vistazo a la reducción de dimensionalidad (PCA)

#### A12 · Ejemplo — Varianza explicada por PCA

Con muchas columnas (tras el one-hot pueden ser decenas), **PCA** las resume en pocas «componentes» que conservan la mayor parte de la variación. No es obligatorio aquí: sirve para visualizar y para comprimir. Se usa solo con columnas numéricas y ya escaladas.

- **Qué hace:**
    - `PCA(n_components=5, random_state=SEMILLA).fit(num_train)` calcula las 5 componentes que más variación recogen de las columnas numéricas escaladas de train.
    - `pca.explained_variance_ratio_` dice qué fracción de la variación recoge cada componente; las celdas de `plt` dibujan un gráfico de barras con esos 5 valores.
- **Qué debe salir:**
    - Un gráfico de 5 barras: ≈ 0,178 · 0,126 · 0,118 · 0,114 · 0,112 (suman ≈ 0,648).
- **Si falla:**
    - `NameError: name 'num_train' is not defined` → falta ejecutar la celda «Imputar y escalar con los datos de entrenamiento» (sección 3).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué componente explica más variación y cuánta, aproximadamente?* → La **primera**, ≈ 0,18 (el 18 %).
2. *¿Qué parte de la variación recogen entre las cinco?* → ≈ **0,65** (65 %): con 5 componentes se conserva casi dos tercios de lo que había en 9 columnas.
3. *¿Por qué las barras son tan parecidas entre sí?* → Porque las columnas numéricas **apenas están correlacionadas** entre sí: no hay una dirección que concentre la variación, así que PCA no puede resumir mucho.

</details>

## 8. Mini-reto: comprueba que tu pipeline es reproducible

#### A13 · Mini-reto — Pipeline reproducible

- **Qué hace:**
    - `clone(modelo)` crea un pipeline **nuevo** con la misma configuración pero **sin ajustar** (no conserva lo aprendido).
    - `.fit(X_train, y_train)` lo ajusta con los mismos datos que `modelo`.
    - `np.array_equal(modelo.predict(X_test), modelo2.predict(X_test))` compara las dos predicciones; `bool(...)` lo deja como verdadero o falso en `son_iguales`.
- **Qué debe salir:**
    - `✔ Mini-reto correcto`. `son_iguales` vale `True`.
- **Si falla:**
    - `Las predicciones deberían coincidir...` → alguna parte es aleatoria y no tiene semilla (`random_state=SEMILLA`) o has ajustado con datos distintos.
    - `modelo2 debe ser una instancia nueva (usa clone)` → has hecho `modelo2 = modelo`: son el mismo objeto. Usa `clone(modelo)`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué vale `son_iguales` y por qué?* → **`True`**: mismos datos, misma configuración y semilla fija ⇒ el mismo ajuste y las mismas predicciones. Eso es un pipeline **reproducible**.
2. *¿Qué hace `clone` y por qué no basta con `modelo2 = modelo`?* → `clone` copia la **configuración** sin lo aprendido, y hay que volver a ajustarlo. `modelo2 = modelo` solo da otro nombre al **mismo objeto**: comparar un modelo consigo mismo no demuestra nada.

</details>

## 9. Para reflexionar

1. ¿Por qué imputamos con la mediana de **train** y no con la de todo el conjunto?
2. ¿Qué pasa si en test aparece un país que no estaba en train? ¿Qué hace `handle_unknown="ignore"`?
3. ¿Cuándo usarías `MinMaxScaler` en vez de `StandardScaler`? ¿Qué le pasa a MinMax con un precio de 9999 €?
4. ¿Qué ventaja tiene guardar todo el preprocesado dentro del pipeline cuando el cliente quiera usar el modelo en producción?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque la mediana de todo el conjunto incluye datos de test: sería fuga (el modelo habría «visto» parte del examen) y, además, en producción los datos nuevos aún no existen. Aquí la mediana del precio es 93,65 con train y 93,25 con todo: poca diferencia, pero el hábito importa.
2. Sin `handle_unknown="ignore"` el codificador daría un **error**. Con esa opción codifica el país nuevo con **ceros en todas las columnas** y el modelo sigue funcionando.
3. `MinMaxScaler` cuando necesitas un rango acotado 0–1 (algunas redes, ciertas visualizaciones) **y no hay valores extremos**. Con un precio de 9999 € el máximo pasaría a ser 9999 y los precios normales (50–250 €) quedarían apretados entre 0 y 0,02, casi sin distinguirse.
4. Es **un único objeto** que recibe los datos crudos y hace imputar, escalar, codificar y predecir con lo aprendido en train: se guarda y se carga entero, no hay que repetir pasos a mano ni olvidar ninguno, y se evitan desajustes y fugas.

</details>

## 10. Resumen: qué has aprendido
- `train_test_split` con estratificación y semilla.
- Imputar, escalar y codificar siempre **con `fit` solo en train**.
- `ColumnTransformer` + `Pipeline` = una receta única, reproducible y sin fuga.